# Dóng hàng theo bài

Sửa khối cấu hình ở đầu cell của tác phẩm rồi chạy bằng kernel `.venv`. Mỗi cell đọc TXT Hán và Việt trong `03_text_sach/`.

1. **Bắc Sứ thông lục (CPO_09).** Batch gồm các đoạn đối chiếu được ở quyển 1 và quyển 4 của TV02. Các khoảng trong `KHOANG_DONG_THEO_MUC` tính từ dòng 1, bỏ qua dòng trống và tính cả hai đầu. Để thử riêng nhật ký ngày 15–16 tháng 7, đặt `CHAY_HANG_LOAT = False`, `SO_QUYEN = 4` và `SO_MUC = 1`.
2. **Văn sách Thi Đình (CPO_11).** Đổi `SO_BAI` trong phạm vi 14–18 của TV02. Mặc định dùng văn xuôi.
3. **Thơ văn Lý–Trần (CPO_12).** Đổi `SO_BAI` trong phạm vi 131–207 của TV02. Chọn `THE_LOAI` theo bài.

- `CHAY_HANG_LOAT = False` chạy một bài hoặc mục và hiện bảng xem trước như cũ.
- `CHAY_HANG_LOAT = True` chạy lần lượt các mục trong `DANH_SACH_MUC` hoặc các bài trong `DANH_SACH_BAI`. Có thể chọn vài bài, ví dụ `[131, 135]`. Mỗi bài ghi vào thư mục duyệt riêng.
- `TIEU_DE_THEO_BAI` ghi số dòng đầu cần bỏ ở từng bản. Bài chưa khai báo giữ toàn bộ dòng. Với Lý–Trần, đặt thể loại riêng ở `THE_LOAI_THEO_BAI` khi khác `THE_LOAI`.

- `CHE_DO = 'nguon'` xem trước tối đa 20 dòng mỗi bản, có số dòng. Điền `TU_KHOA` để tìm trong toàn bộ TXT.
- `CHE_DO = 'nhap'` tạo bản nháp và lưu cấu hình vào `draft.json`.
- `CHE_DO = 'xuat'` xuất bản đã duyệt theo cấu hình lưu cùng bản nháp.

Hán–âm dùng code Lab cùng hai từ điển. Hán–nghĩa dùng API CLC và ghi các liên kết vào `draft.json`.

### Dóng dịch thơ qua Codex

1. Chạy `CHE_DO = 'nhap'`. Mỗi bài có bản dịch thơ sẽ sinh `poetry_prompt.json` trong thư mục duyệt in ở output.
2. Đính kèm file `poetry_prompt.json` đó vào chat Codex hoặc dán toàn bộ nội dung file. Dùng yêu cầu sau.

> Đọc `poetry_prompt.json` và ghép bản Hán với bản dịch thơ có sẵn theo ý. Giữ nguyên `work` và `fingerprint`. Trả JSON có `links`, mỗi nhóm gồm `han_ids`, `tho_ids` và `note`. Chỉ dùng ID trong file, giữ đúng thứ tự, mỗi ID dùng tối đa một lần. Bỏ cặp chưa đủ căn cứ.

3. Lưu phần JSON Codex trả về thành `poetry_result.json` trong cùng thư mục với `poetry_prompt.json`. Chạy lại bài đó với `CHE_DO = 'nhap'` để nạp kết quả vào cột Dịch thơ.

Để xuất, sao chép `review_template.json` thành `review.json`, duyệt `links`, ghi lý do trong `skipped` cho từng ID còn sót và đặt `approved` thành `true`. Chạy lại cell của đúng bài hoặc mục với `CHE_DO = 'xuat'`. Kết quả trong `04_dong_hang/` gồm TSV UTF-8, XLSX có sheet `DONG_HANG` và `notes.xlsx`.

**CLC.** Chạy cell **Nạp token CLC** bên dưới rồi dán token vào ô nhập ẩn hiện ra. Cả ba cell dóng hàng dùng chung `CLC_API_KEY` trong kernel. Phản hồi được lưu vào `clc_response.json` trong thư mục duyệt và dùng lại khi đầu vào giống lần trước. Đổi tên file này nếu cần gửi lại cùng đầu vào. [Tài liệu CLC, trang 1 và 11–13](../guideline/TaiLieuAPI_HanNom_CLC.pdf)

Mỗi lượt tạo bản nháp có dịch nghĩa dùng một request. Bộ đếm nghỉ ít nhất 3 giây và giới hạn 25/phút, 50/giờ, 200/ngày, 600/tuần.

API dùng `requests` trong kernel `.venv`.


In [ ]:
# Nạp token CLC
import os
from getpass import getpass

def nap_token_clc():
    token = os.environ.get('CLC_API_KEY', '').strip()
    if not token:
        print('Dán token vào ô nhập ẩn của notebook rồi nhấn Enter.')
        token = getpass('Token Kim Hán Nôm: ').strip()
    if not token or any(ky_tu.isspace() for ky_tu in token):
        raise ValueError('Nhập đúng phần giá trị của token Kim Hán Nôm.')
    os.environ['CLC_API_KEY'] = token
    print('Đã nạp token CLC. Chạy cell dóng hàng của tác phẩm bạn chọn.')

nap_token_clc()


In [ ]:
# 1. Bắc Sứ thông lục
THANH_VIEN = 'TV02'
CHAY_HANG_LOAT = True
SO_QUYEN = 4
SO_MUC = 1
KHOANG_DONG_THEO_MUC = {
    # Quyển 1 bắt đầu từ tờ biểu của vua An Nam. Hán dòng 1–31 là danh sách quan lại đứng trước phần Việt.
    (1, 1): {'han': (32, 145), 'nghia': (1, 114)},    # Các tờ biểu và tấu tuế cống
    (1, 2): {'han': (146, 259), 'nghia': (115, 225)}, # Quốc tử giám đến khải của Lê Quý Đôn
    (1, 3): {'han': (260, 332), 'nghia': (226, 320)}, # Tư báo Tả giang đạo và phu dịch Lạng Sơn
    (1, 4): {'han': (333, 431), 'nghia': (321, 472)}, # Chọn ngày, cấp phu, chuẩn bị lễ tiễn
    (1, 5): {'han': (432, 565), 'nghia': (473, 657)}, # Công văn tháng 12 đến phần cuối quyển 1
    # Quyển 4 chạy theo thứ tự nhật ký. Mã (4, 1) là mục thử đã dùng trước.
    (4, 2): {'han': (1, 43), 'nghia': (1, 51)},      # Đầu phần trích đến ngày 14 tháng 7
    (4, 1): {'han': (44, 47), 'nghia': (52, 55)},    # Ngày 15–16 tháng 7
    (4, 3): {'han': (48, 83), 'nghia': (56, 115)},   # Ngày 17–30 tháng 7
    (4, 4): {'han': (84, 118), 'nghia': (116, 189)}, # Ngày 1–5 tháng 8
    (4, 5): {'han': (119, 131), 'nghia': (190, 207)}, # Ngày 6–14 tháng 8 đến chỗ Chánh sứ bị ốm
    # Hán dòng 132 là chữ 五 đứng riêng. Việt dòng 208–209 tiếp sang câu chưa có trong phần Hán TV02.
}
DANH_SACH_MUC = list(KHOANG_DONG_THEO_MUC)  # Cả hai quyển theo thứ tự trên
CHE_DO = 'nhap'  # 'nguon', 'nhap' hoặc 'xuat'
TU_KHOA = ''  # Dùng trong chế độ 'nguon'

import sys
from importlib import reload
from pathlib import Path

here = Path.cwd().resolve()
project = next((p for p in (here, *here.parents) if (p / 'ten_file.py').is_file()), None)
if project is None:
    raise FileNotFoundError('Mở notebook trong thư mục project có ten_file.py.')
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from cong_cu import alignment_backends
reload(alignment_backends)
from cong_cu import dong_hang
reload(dong_hang)
from cong_cu import alignment_notebook
reload(alignment_notebook)

muc_duoc_chon = DANH_SACH_MUC if CHAY_HANG_LOAT else [(SO_QUYEN, SO_MUC)]
cau_hinh = [
    {'work': f'quyen{quyen:02d}', 'section': f'muc{muc:02d}',
     'ranges': KHOANG_DONG_THEO_MUC.get((quyen, muc))}
    for quyen, muc in muc_duoc_chon
]
if CHAY_HANG_LOAT:
    bac_su = alignment_notebook.run_book_batch(
        project, 'CPO_09', cau_hinh, member=THANH_VIEN, mode=CHE_DO, keyword=TU_KHOA,
    )
else:
    bac_su = alignment_notebook.run_book_cell(
        project, 'CPO_09', **cau_hinh[0], member=THANH_VIEN, mode=CHE_DO, keyword=TU_KHOA,
    )


In [ ]:
# 2. Văn sách Thi Đình
THANH_VIEN = 'TV02'
CHAY_HANG_LOAT = True
SO_BAI = 14  # 14–18 của TV02
DANH_SACH_BAI = list(range(14, 19))  # Hoặc chọn [14, 15]
THE_LOAI = 'prose'
TIEU_DE_THEO_BAI = {
    14: {'han': 1, 'am': 1, 'nghia': 1},  # Số dòng đầu cần bỏ ở từng bản
}
CHE_DO = 'nhap'  # 'nguon', 'nhap' hoặc 'xuat'
TU_KHOA = ''  # Dùng trong chế độ 'nguon'

import sys
from importlib import reload
from pathlib import Path

here = Path.cwd().resolve()
project = next((p for p in (here, *here.parents) if (p / 'ten_file.py').is_file()), None)
if project is None:
    raise FileNotFoundError('Mở notebook trong thư mục project có ten_file.py.')
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from cong_cu import alignment_backends
reload(alignment_backends)
from cong_cu import dong_hang
reload(dong_hang)
from cong_cu import alignment_notebook
reload(alignment_notebook)

bai_duoc_chon = DANH_SACH_BAI if CHAY_HANG_LOAT else [SO_BAI]
cau_hinh = [
    {'work': f'bai{bai:04d}', 'genre': THE_LOAI, 'titles': TIEU_DE_THEO_BAI.get(bai)}
    for bai in bai_duoc_chon
]
if CHAY_HANG_LOAT:
    thi_dinh = alignment_notebook.run_book_batch(
        project, 'CPO_11', cau_hinh, member=THANH_VIEN, mode=CHE_DO, keyword=TU_KHOA,
    )
else:
    thi_dinh = alignment_notebook.run_book_cell(
        project, 'CPO_11', **cau_hinh[0], member=THANH_VIEN, mode=CHE_DO, keyword=TU_KHOA,
    )


In [ ]:
# 3. Thơ văn Lý–Trần
THANH_VIEN = 'TV02'
CHAY_HANG_LOAT = True
SO_BAI = 135  # 131–207 của TV02
DANH_SACH_BAI = list(range(131, 208))  # Hoặc chọn [131, 135]
THE_LOAI = 'poetry'  # 'prose' nếu bài là văn xuôi
THE_LOAI_THEO_BAI = {}  # Ví dụ {135: 'prose'} nếu bài đó là văn xuôi
TIEU_DE_THEO_BAI = {
    131: {'han': 2, 'am': 3, 'nghia': 2},  # Số dòng đầu cần bỏ ở từng bản
    135: {'han': 1, 'am': 2, 'nghia': 1},
}
CHE_DO = 'nhap'  # 'nguon', 'nhap' hoặc 'xuat'
TU_KHOA = ''  # Dùng trong chế độ 'nguon'

import sys
from importlib import reload
from pathlib import Path

here = Path.cwd().resolve()
project = next((p for p in (here, *here.parents) if (p / 'ten_file.py').is_file()), None)
if project is None:
    raise FileNotFoundError('Mở notebook trong thư mục project có ten_file.py.')
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from cong_cu import alignment_backends
reload(alignment_backends)
from cong_cu import dong_hang
reload(dong_hang)
from cong_cu import alignment_notebook
reload(alignment_notebook)

bai_duoc_chon = DANH_SACH_BAI if CHAY_HANG_LOAT else [SO_BAI]
cau_hinh = [
    {'work': f'bai{bai:04d}', 'genre': THE_LOAI_THEO_BAI.get(bai, THE_LOAI),
     'titles': TIEU_DE_THEO_BAI.get(bai)}
    for bai in bai_duoc_chon
]
if CHAY_HANG_LOAT:
    ly_tran = alignment_notebook.run_book_batch(
        project, 'CPO_12', cau_hinh, member=THANH_VIEN, mode=CHE_DO, keyword=TU_KHOA,
    )
else:
    ly_tran = alignment_notebook.run_book_cell(
        project, 'CPO_12', **cau_hinh[0], member=THANH_VIEN, mode=CHE_DO, keyword=TU_KHOA,
    )


In [ ]:
# 4. Gộp bảng dóng hàng để nộp
THANH_VIEN = "TV02"
TAC_PHAM_CHON = ["CPO_11", "CPO_12"]  # Ví dụ chỉ lấy Thi Đình: ["CPO_11"]

import csv
import io
import re
import sys
from pathlib import Path

here = Path.cwd().resolve()
project = next((p for p in (here, *here.parents) if (p / "ten_file.py").is_file()), None)
if project is None:
    raise FileNotFoundError("Mở notebook trong repo có ten_file.py.")
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from cong_cu.dong_hang import HEADERS, clean_book_root, write_protected, xlsx_bytes

if not TAC_PHAM_CHON or len(set(TAC_PHAM_CHON)) != len(TAC_PHAM_CHON):
    raise ValueError("Chọn ít nhất một mã tác phẩm và mỗi mã chỉ xuất hiện một lần.")
if any(not re.fullmatch(r"CPO_\d{2}", code) for code in TAC_PHAM_CHON):
    raise ValueError("Mã tác phẩm phải có dạng CPO_11 hoặc CPO_12.")

rows = []
table_count = 0
for code in TAC_PHAM_CHON:
    source = clean_book_root(project, code, THANH_VIEN) / "04_dong_hang"
    tables = {}
    for extension in ("csv", "tsv"):
        for file in source.glob(f"*/{code}_*_parallel.{extension}"):
            tables[file.with_suffix("")] = file
    if not tables:
        raise FileNotFoundError(f"{code} chưa có bảng dóng hàng đã xuất ở {source}.")

    for file in sorted(tables.values()):
        with file.open(encoding="utf-8-sig", newline="") as stream:
            if file.suffix == ".tsv":
                work_rows = list(csv.reader(stream, delimiter="\t", quoting=csv.QUOTE_NONE))
            else:
                work_rows = list(csv.reader(stream, strict=True))
                if work_rows and tuple(work_rows[0]) == HEADERS:
                    work_rows = work_rows[1:]
        if not work_rows or any(len(row) != len(HEADERS) for row in work_rows):
            raise ValueError(f"Bảng phải có dữ liệu và đúng 4 cột: {file}")
        rows.extend(work_rows)
    table_count += len(tables)

output = project / "san_pham" / THANH_VIEN / "04_dong_hang"
name = f"{THANH_VIEN}_{'_'.join(TAC_PHAM_CHON)}_parallel"
csv_text = io.StringIO(newline="")
writer = csv.writer(csv_text)
writer.writerow(HEADERS)
writer.writerows(rows)
files = {
    output / f"{name}.csv": csv_text.getvalue().encode("utf-8-sig"),
    output / f"{name}.xlsx": xlsx_bytes("DONG_HANG", HEADERS, rows),
}
write_protected(files)

print(f"Đã gộp {table_count} bảng, tổng {len(rows)} hàng dữ liệu.")
for path in files:
    print(path)
